# 01 — Exploration : le Bénin et l'Afrique de l'Ouest (1990–2024)

Données : **World Development Indicators** (Banque mondiale, CC BY 4.0), 16 pays d'Afrique de l'Ouest + agrégats Afrique subsaharienne et Monde. Instantané téléchargé via l'API (voir `data/raw/metadata.json`).

**Questions**
1. Où se situe le Bénin face à ses voisins aujourd'hui ?
2. Comment la santé, l'électricité et Internet ont-ils évolué depuis 2000 ?
3. Quels indicateurs évoluent avec l'espérance de vie ?

In [1]:
import sys
from pathlib import Path
ROOT = Path.cwd().resolve().parent if Path.cwd().name == "notebooks" else Path.cwd().resolve()
sys.path.insert(0, str(ROOT))
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
%matplotlib inline
pd.set_option("display.precision", 2)
from src.config import COUNTRIES, FOCUS, LABELS
from src.data import load_panel
from src import analysis, plots
panel = load_panel()
countries = panel[~panel.is_aggregate]
print(panel.shape)
print(panel.year.min(), '–', panel.year.max(), '·', countries.iso3.nunique(), 'pays')
panel.head()

(630, 16)
1990 – 2024 · 16 pays


,iso3,country,is_aggregate,year,life_expectancy,gdp_pc,gdp_growth,population,urban_pct,fertility,under5_mortality,electricity,internet,mobile,water,primary_completion
0,BEN,Bénin,False,1990,53.02,749.46,8.98,5.28e+06,34.82,6.74,171.2,NaN,0.0,0.0,NaN,17.64
1,BEN,Bénin,False,1991,53.44,757.40,4.23,5.45e+06,35.34,6.67,167.0,NaN,NaN,0.0,NaN,19.87
2,BEN,Bénin,False,1992,54.14,756.08,2.96,5.62e+06,35.79,6.59,163.0,NaN,NaN,0.0,NaN,19.82
3,BEN,Bénin,False,1993,54.61,765.48,5.84,5.87e+06,36.18,6.51,159.0,NaN,NaN,0.0,NaN,21.61
4,BEN,Bénin,False,1994,54.99,752.28,2.02,6.10e+06,36.53,6.42,155.2,NaN,NaN,0.0,NaN,23.39


## 1. Qualité des données
Part de valeurs manquantes par indicateur (pays uniquement, 1990–2024). L'achèvement du primaire est très lacunaire : il est exclu des analyses principales.

In [2]:
(countries[list(LABELS)].isna().mean() * 100).round(1).sort_values().rename('% manquant').to_frame()

,% manquant
life_expectancy,0.0
gdp_pc,0.0
gdp_growth,0.0
population,0.0
urban_pct,0.0
fertility,0.0
under5_mortality,0.0
mobile,2.3
internet,12.5
electricity,18.9


## 2. Le Bénin face à ses voisins (dernière année complète)

In [3]:
from src.report import compute
key, corr = compute(panel)
year = key['reference_year']
snap = countries[countries.year == year]
cols = ['life_expectancy', 'gdp_pc', 'electricity', 'internet', 'water', 'fertility', 'under5_mortality', 'urban_pct']
table = snap.set_index('country')[cols].rename(columns=lambda c: LABELS[c].split(' (')[0])
print('Année de référence :', year)
table.loc[['Bénin', 'Togo', 'Nigeria', 'Niger', 'Burkina Faso', 'Ghana', "Côte d'Ivoire", 'Sénégal']].round(1)

Année de référence : 2024


,Espérance de vie à la naissance,PIB par habitant,Accès à l'électricité,Utilisateurs d'Internet,Accès à l'eau potable de base,Fécondité,Mortalité des moins de 5 ans,Population urbaine
country,,,,,,,,
Bénin,61.0,1324.8,59.0,34.0,69.6,4.5,74.7,52.5
Togo,62.9,1053.8,61.1,39.5,64.8,4.1,56.1,43.8
Nigeria,54.6,2324.6,62.5,41.2,82.2,4.4,115.6,63.0
Niger,61.4,589.1,21.3,15.6,53.4,5.9,110.7,18.0
Burkina Faso,61.3,764.1,34.0,28.3,50.0,4.1,74.9,28.3
Ghana,65.7,2173.0,91.9,72.2,89.9,3.3,35.9,58.4
Côte d'Ivoire,62.1,2395.7,72.9,41.4,77.2,4.2,64.5,54.0
Sénégal,68.9,1517.6,82.9,60.1,88.0,3.8,36.5,55.2


In [4]:
pd.DataFrame({c: {'Bénin': v['benin'], 'rang Bénin /16': v['benin_rank_of_16'], 'médiane Afrique de l\'Ouest': v['west_africa_median'],
                   'Afrique subsaharienne': v['ssf']} for c, v in key['indicators'].items()}).T.round(1)

,Bénin,rang Bénin /16,médiane Afrique de l'Ouest,Afrique subsaharienne
life_expectancy,61.0,13.0,62.2,62.8
gdp_pc,1324.8,7.0,1095.6,1588.9
electricity,59.0,8.0,56.4,55.1
internet,34.0,10.0,38.1,33.6
fertility,4.5,4.0,4.1,4.3
under5_mortality,74.7,7.0,70.2,71.2
urban_pct,52.5,9.0,53.3,44.1
water,69.6,11.0,77.3,67.5


In [5]:
gdp = analysis.latest(panel[(panel.year <= year) & panel.iso3.isin(list(COUNTRIES) + ['SSF'])], 'gdp_pc')
plots.ranking(gdp, 'gdp_pc', f'PIB par habitant en {year} (USD constants 2015)');
print(f"Croissance annuelle moyenne du PIB/hab. du Bénin 2000–{year} : {key['benin_gdp_pc_cagr_2000_last']:.2f} %")

Croissance annuelle moyenne du PIB/hab. du Bénin 2000–2024 : 1.81 %


## 3. Trajectoires depuis 1990

In [6]:
plots.trend(panel, 'life_expectancy', f'Espérance de vie à la naissance, 1990–{year}', ylabel='Années');

In [7]:
le = analysis.change(countries, 'life_expectancy', 2000, year)
le.round(1)

,country,start,end,change
iso3,,,,
SLE,Sierra Leone,44.8,62.0,17.1
GNB,Guinée-Bissau,50.6,64.3,13.7
NER,Niger,49.2,61.4,12.3
SEN,Sénégal,57.6,68.9,11.3
CIV,Côte d'Ivoire,51.4,62.1,10.7
BFA,Burkina Faso,50.8,61.3,10.5
LBR,Libéria,51.9,62.3,10.5
MLI,Mali,50.6,60.7,10.1
GMB,Gambie,56.9,66.1,9.2


Le Bénin partait d'un niveau relativement élevé en 2000, mais affiche le **plus petit gain** des 16 pays sur la période : les pays qui partaient de très bas (Sierra Leone, Guinée-Bissau, Niger…) ont rattrapé une partie de leur retard.

In [8]:
el = analysis.change(panel[panel.iso3.isin(list(COUNTRIES) + ['SSF'])], 'electricity', 2000, year)
plots.dumbbell(el.dropna(), f"Accès à l'électricité : 2000 → {year}", '% de la population', start_label='2000', end_label=str(year));
el.round(1)

,country,start,end,change
iso3,,,,
GHA,Ghana,43.7,91.9,48.2
SEN,Sénégal,37.7,82.9,45.2
TGO,Togo,17.0,61.1,44.1
MLI,Mali,9.6,49.5,39.9
CPV,Cabo Verde,58.6,98.4,39.8
GIN,Guinée,15.2,53.8,38.6
BEN,Bénin,21.5,59.0,37.5
GMB,Gambie,34.3,69.0,34.7
MRT,Mauritanie,19.1,52.2,33.1


In [9]:
plots.trend(panel, 'internet', f"Utilisateurs d'Internet, 2000–{year}", start=2000, ylabel='% de la population');

In [10]:
ben = panel[panel.iso3 == 'BEN'].set_index('year')
ben.loc[[2000, 2010, 2020, year], ['population', 'urban_pct', 'fertility', 'under5_mortality', 'electricity', 'internet']].round(1)

,population,urban_pct,fertility,under5_mortality,electricity,internet
year,,,,,,
2000,7.22e+06,38.3,5.9,135.0,21.5,0.2
2010,9.80e+06,42.9,5.5,107.4,34.2,3.1
2020,1.31e+07,49.4,4.8,84.6,41.0,22.1
2024,1.45e+07,52.5,4.5,74.7,59.0,34.0


## 4. Corrélations entre indicateurs
Corrélation de Spearman sur toutes les observations pays-année complètes. Attention : ces indicateurs progressent tous avec le temps, une partie des corrélations reflète donc une tendance commune.

In [11]:
plots.heatmap(corr, 'Corrélations entre indicateurs (16 pays, toutes années)');
print('Observations complètes :', key['n_country_years_corr'])
pd.Series(key['spearman_with_life_expectancy'], name='Spearman avec l\'espérance de vie')

Observations complètes : 391


under5_mortality   -0.96
fertility          -0.78
gdp_pc              0.42
electricity         0.56
urban_pct           0.57
water               0.60
internet            0.73
Name: Spearman avec l'espérance de vie, dtype: float64

## À retenir
- Le Bénin est dans le milieu du peloton pour le PIB par habitant et l'électricité, mais en dessous de la médiane régionale pour l'espérance de vie, Internet et l'eau potable.
- Les progrès d'accès à l'électricité et à Internet sont massifs dans toute la région depuis 2000.
- L'espérance de vie est très liée à la mortalité infantile et à la fécondité (corrélations négatives fortes) ; le PIB par habitant n'y est que modérément associé.
- Corrélation ≠ causalité : ces liens sont descriptifs.